# Harmonic Analysis from CPOW Data

This notebook performs FFT-based harmonic analysis on continuous point-on-wave (CPOW)
waveform data. It computes harmonic spectra, Total Harmonic Distortion (THD), and
time-frequency spectrograms.

**Sections:**
1. Setup and load CPOW data
2. Compute FFT
3. Harmonic spectrum
4. THD calculation
5. Spectrogram (time-frequency analysis)
6. Compare phases
7. IEEE 519 context

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline

from equser.data import SAMPLE_RATE_HZ, load_cpow_scaled
from equser.widgets import create_file_selector

SAMPLE_RATE = SAMPLE_RATE_HZ  # 32,000 Hz
NOMINAL_FREQ = 60  # Hz (adjust for 50 Hz systems)

In [ ]:
# Select a CPOW file
cpow_dir = Path('/var/lib/eq-watch/data/cpow')  # Adjust as needed
file_selector = create_file_selector(cpow_dir)
display(file_selector)

In [ ]:
cpow_file = file_selector.get_selected()
# Or override:
# cpow_file = cpow_dir / '20250623_075156.parquet'

print(f"Loading: {cpow_file}")
cpow = load_cpow_scaled(cpow_file)

VA = cpow['VA']
VB = cpow['VB']
VC = cpow['VC']

duration_sec = len(VA) / SAMPLE_RATE
print(f"Duration: {duration_sec:.1f} seconds, {len(VA):,} samples")
print(f"Scaling: vscale={cpow['vscale']}, iscale={cpow['iscale']}")

## 2. Compute FFT

We apply a Hanning window to reduce spectral leakage, then compute the real FFT.
Using a window size that is a multiple of the fundamental period (1/60 s) gives
clean harmonic peaks.

In [ ]:
# Use a window of exactly N fundamental cycles for clean harmonic bins
cycles_per_window = 10
window_samples = int(cycles_per_window * SAMPLE_RATE / NOMINAL_FREQ)

# Take the first window_samples from the signal
segment = VA[:window_samples]
window = np.hanning(len(segment))
windowed = segment * window

# Compute real FFT
fft_result = np.fft.rfft(windowed)
fft_magnitude = np.abs(fft_result) * 2.0 / np.sum(window)  # Normalize for window energy
fft_freq = np.fft.rfftfreq(len(segment), d=1.0 / SAMPLE_RATE)

# Convert to dB (relative to fundamental)
fundamental_idx = np.argmin(np.abs(fft_freq - NOMINAL_FREQ))
fundamental_mag = fft_magnitude[fundamental_idx]
fft_db = 20 * np.log10(fft_magnitude / fundamental_mag + 1e-12)

print(
    f"Window: {window_samples} samples ({window_samples / SAMPLE_RATE * 1000:.1f} ms, {cycles_per_window} cycles)"
)
print(f"Frequency resolution: {fft_freq[1]:.2f} Hz")
print(f"Fundamental magnitude: {fundamental_mag:.1f} V")

In [ ]:
# Plot full spectrum
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

# Linear scale
ax1.set_title('FFT Spectrum (Phase VA) - Linear')
ax1.plot(fft_freq, fft_magnitude, 'k-', linewidth=0.5)
ax1.set_xlabel('Frequency (Hz)')
ax1.set_ylabel('Magnitude (V)')
ax1.set_xlim(0, 3000)
ax1.grid(True, alpha=0.3)

# dB scale
ax2.set_title('FFT Spectrum (Phase VA) - dB relative to fundamental')
ax2.plot(fft_freq, fft_db, 'k-', linewidth=0.5)
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Magnitude (dB)')
ax2.set_xlim(0, 3000)
ax2.set_ylim(-80, 5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Harmonic spectrum

Extract individual harmonic magnitudes (fundamental through 50th harmonic).

In [ ]:
max_harmonic = 50
harmonic_freqs = np.arange(1, max_harmonic + 1) * NOMINAL_FREQ
harmonic_mags = np.zeros(max_harmonic)

for h in range(max_harmonic):
    target_freq = harmonic_freqs[h]
    if target_freq > fft_freq[-1]:
        break
    idx = np.argmin(np.abs(fft_freq - target_freq))
    harmonic_mags[h] = fft_magnitude[idx]

# Normalize to percentage of fundamental
harmonic_pct = harmonic_mags / fundamental_mag * 100

# Print top harmonics
print(f"{'Harmonic':>10} {'Freq (Hz)':>10} {'Magnitude (V)':>15} {'% of Fund.':>12}")
print('-' * 50)
for h in range(min(15, max_harmonic)):
    if harmonic_mags[h] > 0:
        print(
            f"{h + 1:>10} {harmonic_freqs[h]:>10.0f} {harmonic_mags[h]:>15.3f} {harmonic_pct[h]:>11.2f}%"
        )

In [ ]:
# Bar chart of harmonic magnitudes
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Harmonic Spectrum (Phase VA)')

harmonics = np.arange(1, max_harmonic + 1)
colors = [
    'black' if h == 1 else ('red' if harmonic_pct[h - 1] > 5 else 'steelblue') for h in harmonics
]

ax.bar(harmonics, harmonic_pct, color=colors, edgecolor='none', width=0.8)
ax.set_xlabel('Harmonic Number')
ax.set_ylabel('% of Fundamental')
ax.set_xlim(0.5, max_harmonic + 0.5)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. THD calculation

Total Harmonic Distortion is the ratio of the RMS of all harmonic components
to the fundamental:

$$\text{THD} = \frac{\sqrt{\sum_{h=2}^{N} V_h^2}}{V_1} \times 100\%$$

In [ ]:
def compute_thd(signal, sample_rate, nominal_freq, max_harmonic=50, cycles_per_window=10):
    """Compute THD for a voltage or current signal."""
    win_len = int(cycles_per_window * sample_rate / nominal_freq)
    seg = signal[:win_len]
    win = np.hanning(len(seg))
    fft_mag = np.abs(np.fft.rfft(seg * win)) * 2.0 / np.sum(win)
    freqs = np.fft.rfftfreq(len(seg), d=1.0 / sample_rate)

    fund_idx = np.argmin(np.abs(freqs - nominal_freq))
    v1 = fft_mag[fund_idx]

    harmonic_sum_sq = 0.0
    for h in range(2, max_harmonic + 1):
        target = h * nominal_freq
        if target > freqs[-1]:
            break
        idx = np.argmin(np.abs(freqs - target))
        harmonic_sum_sq += fft_mag[idx] ** 2

    thd = np.sqrt(harmonic_sum_sq) / v1 * 100 if v1 > 0 else 0.0
    return thd, v1


# Compute THD for each phase
for name, sig in [('VA', VA), ('VB', VB), ('VC', VC)]:
    thd, v1 = compute_thd(sig, SAMPLE_RATE, NOMINAL_FREQ)
    print(f"{name}: THD = {thd:.2f}%, Fundamental = {v1:.1f} V")

## 5. Spectrogram

A time-frequency heatmap shows how the harmonic content evolves over the duration
of the file. We use a sliding-window FFT with a step size of a few cycles.

In [ ]:
# Spectrogram parameters
window_cycles = 10
step_cycles = 5
window_len = int(window_cycles * SAMPLE_RATE / NOMINAL_FREQ)
step_len = int(step_cycles * SAMPLE_RATE / NOMINAL_FREQ)
max_freq_hz = 3000

# Compute spectrogram
n_windows = (len(VA) - window_len) // step_len + 1
freqs = np.fft.rfftfreq(window_len, d=1.0 / SAMPLE_RATE)
freq_mask = freqs <= max_freq_hz

spectrogram = np.zeros((n_windows, np.sum(freq_mask)))
times = np.zeros(n_windows)
win = np.hanning(window_len)

for i in range(n_windows):
    start = i * step_len
    seg = VA[start : start + window_len]
    fft_mag = np.abs(np.fft.rfft(seg * win)) * 2.0 / np.sum(win)
    spectrogram[i, :] = 20 * np.log10(fft_mag[freq_mask] + 1e-12)
    times[i] = start / SAMPLE_RATE

print(f"Computed {n_windows} windows")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_title('Spectrogram (Phase VA)')

im = ax.pcolormesh(times, freqs[freq_mask], spectrogram.T, shading='auto', cmap='inferno')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Frequency (Hz)')
plt.colorbar(im, ax=ax, label='Magnitude (dB)')

# Mark harmonic frequencies
for h in range(1, 11):
    ax.axhline(h * NOMINAL_FREQ, color='white', alpha=0.2, linewidth=0.5)

plt.tight_layout()
plt.show()

## 6. Compare phases

Overlay the harmonic profiles for all three voltage phases.

In [ ]:
def get_harmonic_profile(signal, sample_rate, nominal_freq, max_h=50, cycles=10):
    """Return harmonic magnitudes as percentage of fundamental."""
    win_len = int(cycles * sample_rate / nominal_freq)
    seg = signal[:win_len]
    win = np.hanning(len(seg))
    fft_mag = np.abs(np.fft.rfft(seg * win)) * 2.0 / np.sum(win)
    freqs = np.fft.rfftfreq(len(seg), d=1.0 / sample_rate)

    fund_idx = np.argmin(np.abs(freqs - nominal_freq))
    v1 = fft_mag[fund_idx]

    mags = []
    for h in range(1, max_h + 1):
        target = h * nominal_freq
        if target > freqs[-1]:
            mags.append(0)
        else:
            idx = np.argmin(np.abs(freqs - target))
            mags.append(fft_mag[idx] / v1 * 100 if v1 > 0 else 0)
    return np.array(mags)


max_h = 25
harmonics = np.arange(1, max_h + 1)
profiles = {
    'VA': get_harmonic_profile(VA, SAMPLE_RATE, NOMINAL_FREQ, max_h),
    'VB': get_harmonic_profile(VB, SAMPLE_RATE, NOMINAL_FREQ, max_h),
    'VC': get_harmonic_profile(VC, SAMPLE_RATE, NOMINAL_FREQ, max_h),
}

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Harmonic Profile Comparison (Voltage Phases)')
width = 0.25
for i, (name, color) in enumerate([('VA', 'black'), ('VB', 'red'), ('VC', 'blue')]):
    ax.bar(
        harmonics + (i - 1) * width, profiles[name], width=width, color=color, alpha=0.7, label=name
    )

ax.set_xlabel('Harmonic Number')
ax.set_ylabel('% of Fundamental')
ax.set_xlim(1.5, max_h + 0.5)  # Skip fundamental for better scale
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 7. IEEE 519 context

[IEEE 519-2022](https://standards.ieee.org/standard/519-2022.html) sets recommended
limits for harmonic distortion at the point of common coupling (PCC). The limits
depend on the system voltage level and the ratio of short-circuit current to load current.

The plot below annotates typical voltage THD limits for reference. These are
**informational only**; actual compliance requires measurements at the PCC with
proper averaging intervals.

| Bus Voltage | Individual Harmonic (%) | THD (%) |
|-------------|------------------------|---------|
| V <= 1 kV   | 5.0                    | 8.0     |
| 1 kV < V <= 69 kV | 3.0              | 5.0     |
| 69 kV < V <= 161 kV | 1.5            | 2.5     |

In [ ]:
# Annotated harmonic bar chart with IEEE 519 reference lines
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_title('Harmonic Spectrum with IEEE 519 Reference (Phase VA, V <= 1 kV)')

ax.bar(harmonics, profiles['VA'], color='steelblue', edgecolor='none', width=0.8)

# IEEE 519 limits for V <= 1 kV
ax.axhline(5.0, color='red', linestyle='--', alpha=0.7, label='IEEE 519 individual limit (5%)')
ax.axhline(8.0, color='darkred', linestyle=':', alpha=0.7, label='IEEE 519 THD limit (8%)')

ax.set_xlabel('Harmonic Number')
ax.set_ylabel('% of Fundamental')
ax.set_xlim(1.5, max_h + 0.5)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()